# 🧠 Binary Classification: Predicting Age Group from LLM Interactions
### Machine Learning Project – Google Colab Notebook

**Goal:** Build the best binary classification model to predict whether a user belongs to the *Young Adults* or *Older Adults* group, based on their interactions with a Large Language Model (ChatGPT).

**Dataset:** `all_tasks_90_sub_23_12.csv` – 1,275 conversation records between users and GPT, with metadata and demographic labels.

**Target Variable:** `subject_group` (Young_Adults vs Older_Adults)

---
**Table of Contents:**
1. Environment Setup & Imports
2. Data Loading & Exploratory Data Analysis
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Feature Selection
6. Train/Test Split & Scaling
7. Model Training & Comparison
8. Hyperparameter Tuning
9. Final Evaluation
10. Summary & Conclusions

## 1. Environment Setup & Imports

### 1.1 Install Required Libraries
We install all necessary packages that are not pre-installed on Google Colab.

In [ ]:
!pip install textstat xgboost nltk scikit-learn transformers torch

import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
print("✅ All packages installed and NLTK data downloaded.")

### 1.2 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, roc_curve, auc)
from xgboost import XGBClassifier
from scipy.stats import loguniform, randint, uniform

# NLP
import textstat
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
import string
from collections import Counter

# Plotting style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully.")

## 2. Data Loading & Exploratory Data Analysis (EDA)

### 2.1 Load the Dataset
Upload the CSV file to Colab's `/content/` directory, then load it into a pandas DataFrame.

In [ ]:
# Load the dataset
df = pd.read_csv('/content/all_tasks_90_sub_23_12.csv')
df_original = df.copy()  # Keep an unmodified copy for reference

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")
df.head()

### 2.2 Data Overview & Statistics

In [ ]:
# Data types and non-null counts
print("="*60)
print("DATA TYPES & NULL COUNTS")
print("="*60)
print(df.dtypes)
print(f"\nTotal missing values per column:")
print(df.isnull().sum())
print(f"\nBasic statistics for numerical columns:")
df.describe()

### 2.3 Target Variable Distribution

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
target_counts = df['subject_group'].value_counts()
colors = ['#2ecc71', '#e74c3c']
target_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subject Group')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.03, 0.03), shadow=True,
            textprops={'fontsize': 12})
axes[1].set_title('Target Variable Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nClass balance ratio: {target_counts.min()/target_counts.max():.2f} — fairly balanced ✅")

### 2.4 Feature Overview

In [ ]:
# Quick look at unique values per column
print("UNIQUE VALUES PER COLUMN:")
print("="*50)
for col in df.columns:
    n_unique = df[col].nunique()
    dtype = df[col].dtype
    sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else 'N/A'
    sample_str = str(sample)[:50] + '...' if len(str(sample)) > 50 else str(sample)
    print(f"  {col:45s} | {str(dtype):10s} | {n_unique:5d} unique | e.g. {sample_str}")

# Numerical features distribution
num_cols = ['msg_count_within_p', 'Age', 'response_time_sec', 'response_time_min', 'words_in_message_to_gpt']
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))
for i, col in enumerate(num_cols):
    df.boxplot(column=col, by='subject_group', ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
plt.suptitle('Numerical Features by Subject Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Preprocessing

### 3.1 Drop ID & Timestamp Columns
These columns are identifiers with no predictive value. We also encode the target variable.

> **Important Note on `Age`:** The `Age` feature has ~0.97 correlation with the target variable `subject_group`. This is expected since the groups *are* defined by age ranges, making `Age` essentially a label proxy. **We exclude `Age` from all models** to ensure the model learns meaningful behavioral patterns, not just age thresholds.

In [ ]:
# Columns to drop (IDs, timestamps, raw text handled separately, and Age)
columns_to_drop = [
    'msg_id', 'gpt_interface_id', 'participant_id', 'subject_id',
    'unix_time_when_msg_sent_to_gpt', 'date_time_when_msg_sent_to_gpt',
    'unix_time_when_msg_received_from_gpt', 'date_time_when_msg_received_from_gpt',
]

# Encode target variable
df['target'] = df['subject_group'].map({'Young_Adults': 0, 'Older_Adults': 1})
print(f"Target encoding: Young_Adults → 0, Older_Adults → 1")
print(f"Target distribution:\n{df['target'].value_counts()}")

# One-hot encode 'Sex'
print(f"\nSex values: {df['Sex'].unique()}")
df = pd.get_dummies(df, columns=['Sex'], prefix='Sex', drop_first=False)

# One-hot encode 'TASK'
print(f"TASK values: {df['TASK'].unique()}")
df = pd.get_dummies(df, columns=['TASK'], prefix='TASK', drop_first=False)

# Store text columns before dropping
text_to_gpt = df['message_to_gpt'].astype(str).fillna('')
text_from_gpt = df['message_from_gpt'].astype(str).fillna('')

# Drop unnecessary columns
df.drop(columns=columns_to_drop + ['subject_group', 'message_to_gpt', 'message_from_gpt', 'Age'], 
        inplace=True, errors='ignore')

# Convert boolean columns to int
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

print(f"\nRemaining columns ({df.shape[1]}): {list(df.columns)}")

## 4. Feature Engineering

### 4.1 Text Length & Basic Text Features
We extract simple but informative features from the message columns.

In [ ]:
# Text length features
df['msg_to_gpt_char_count'] = text_to_gpt.str.len()
df['msg_to_gpt_word_count'] = text_to_gpt.str.split().str.len()
df['msg_from_gpt_char_count'] = text_from_gpt.str.len()
df['msg_from_gpt_word_count'] = text_from_gpt.str.split().str.len()
df['msg_to_gpt_avg_word_len'] = df['msg_to_gpt_char_count'] / (df['msg_to_gpt_word_count'] + 1)
df['msg_from_gpt_avg_word_len'] = df['msg_from_gpt_char_count'] / (df['msg_from_gpt_word_count'] + 1)

# Ratio features
df['response_word_ratio'] = df['msg_from_gpt_word_count'] / (df['msg_to_gpt_word_count'] + 1)

print(f"Added {7} text length features")
df[['msg_to_gpt_char_count', 'msg_to_gpt_word_count', 'msg_from_gpt_char_count',
    'msg_from_gpt_word_count', 'msg_to_gpt_avg_word_len', 'response_word_ratio']].describe()

### 4.2 Sentiment Analysis (VADER)
We extract sentiment polarity scores from both message columns using NLTK's VADER sentiment analyzer.

In [ ]:
sid = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    scores = sid.polarity_scores(str(text))
    return scores['compound'], scores['pos'], scores['neg'], scores['neu']

# Apply VADER to message_to_gpt
vader_to = text_to_gpt.apply(lambda x: get_vader_scores(x))
df['msg_to_gpt_sentiment_compound'] = vader_to.apply(lambda x: x[0])
df['msg_to_gpt_sentiment_pos'] = vader_to.apply(lambda x: x[1])
df['msg_to_gpt_sentiment_neg'] = vader_to.apply(lambda x: x[2])

# Apply VADER to message_from_gpt
vader_from = text_from_gpt.apply(lambda x: get_vader_scores(x))
df['msg_from_gpt_sentiment_compound'] = vader_from.apply(lambda x: x[0])
df['msg_from_gpt_sentiment_pos'] = vader_from.apply(lambda x: x[1])
df['msg_from_gpt_sentiment_neg'] = vader_from.apply(lambda x: x[2])

print("✅ Added 6 sentiment features")
df[['msg_to_gpt_sentiment_compound', 'msg_from_gpt_sentiment_compound']].describe()

### 4.3 Readability Scores
We calculate readability metrics for the user's messages to capture writing complexity differences between age groups.

In [ ]:
# Readability features for message_to_gpt
df['msg_to_gpt_flesch_kincaid'] = text_to_gpt.apply(textstat.flesch_kincaid_grade)
df['msg_to_gpt_flesch_ease'] = text_to_gpt.apply(textstat.flesch_reading_ease)
df['msg_to_gpt_ari'] = text_to_gpt.apply(textstat.automated_readability_index)

# Readability features for message_from_gpt
df['msg_from_gpt_flesch_kincaid'] = text_from_gpt.apply(textstat.flesch_kincaid_grade)
df['msg_from_gpt_flesch_ease'] = text_from_gpt.apply(textstat.flesch_reading_ease)

print("✅ Added 5 readability features")
df[['msg_to_gpt_flesch_kincaid', 'msg_to_gpt_flesch_ease', 'msg_to_gpt_ari']].describe()

### 4.4 TF-IDF Features with Dimensionality Reduction
We extract TF-IDF n-gram features from `message_to_gpt` and reduce dimensionality using TruncatedSVD (LSA).

In [ ]:
# TF-IDF for message_to_gpt (unigrams + bigrams)
tfidf_to = TfidfVectorizer(ngram_range=(1, 2), max_features=500, min_df=5, stop_words='english')
tfidf_to_matrix = tfidf_to.fit_transform(text_to_gpt)

# Reduce to 20 components using TruncatedSVD (Latent Semantic Analysis)
svd_to = TruncatedSVD(n_components=20, random_state=42)
tfidf_to_reduced = svd_to.fit_transform(tfidf_to_matrix)
tfidf_to_df = pd.DataFrame(tfidf_to_reduced, columns=[f'tfidf_to_svd_{i}' for i in range(20)], index=df.index)

print(f"TF-IDF message_to_gpt: {tfidf_to_matrix.shape[1]} features → 20 SVD components")
print(f"  Explained variance: {svd_to.explained_variance_ratio_.sum():.2%}")

# TF-IDF for message_from_gpt
tfidf_from = TfidfVectorizer(ngram_range=(1, 2), max_features=500, min_df=5, stop_words='english')
tfidf_from_matrix = tfidf_from.fit_transform(text_from_gpt)

svd_from = TruncatedSVD(n_components=20, random_state=42)
tfidf_from_reduced = svd_from.fit_transform(tfidf_from_matrix)
tfidf_from_df = pd.DataFrame(tfidf_from_reduced, columns=[f'tfidf_from_svd_{i}' for i in range(20)], index=df.index)

print(f"TF-IDF message_from_gpt: {tfidf_from_matrix.shape[1]} features → 20 SVD components")
print(f"  Explained variance: {svd_from.explained_variance_ratio_.sum():.2%}")

# Add to main dataframe
df = pd.concat([df, tfidf_to_df, tfidf_from_df], axis=1)
print(f"\n✅ DataFrame shape after TF-IDF features: {df.shape}")

### 4.5 NMF Topic Features
Non-negative Matrix Factorization extracts interpretable topic features from the TF-IDF matrices.

In [ ]:
# NMF topics from message_to_gpt
n_topics = 10
nmf_to = NMF(n_components=n_topics, random_state=42)
nmf_to_features = nmf_to.fit_transform(tfidf_to_matrix)
nmf_to_df = pd.DataFrame(nmf_to_features, columns=[f'nmf_to_topic_{i}' for i in range(n_topics)], index=df.index)

# NMF topics from message_from_gpt
nmf_from = NMF(n_components=n_topics, random_state=42)
nmf_from_features = nmf_from.fit_transform(tfidf_from_matrix)
nmf_from_df = pd.DataFrame(nmf_from_features, columns=[f'nmf_from_topic_{i}' for i in range(n_topics)], index=df.index)

df = pd.concat([df, nmf_to_df, nmf_from_df], axis=1)

# Display top words per topic for message_to_gpt
feature_names = tfidf_to.get_feature_names_out()
print("Top 5 words per NMF topic (message_to_gpt):")
for i, topic in enumerate(nmf_to.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-5:][::-1]]
    print(f"  Topic {i}: {', '.join(top_words)}")

print(f"\n✅ Added {n_topics * 2} NMF topic features. DataFrame shape: {df.shape}")

### 4.6 Lexical Diversity & Linguistic Style Features
These NLP features capture **how** users write, not just what they write:
- **Type-Token Ratio (TTR):** vocabulary richness — ratio of unique words to total words
- **Hapax Legomenon Ratio:** proportion of words used only once (rarer = more diverse)
- **Punctuation density:** use of commas, periods, exclamation marks, question marks
- **POS tag distributions:** proportions of nouns, verbs, adjectives, adverbs
- **Function word ratio:** proportion of common function words (a, the, is, etc.)

In [ ]:
def compute_lexical_features(text):
    text = str(text)
    words = text.lower().split()
    n_words = len(words) if len(words) > 0 else 1
    
    # Type-Token Ratio (TTR) — lexical diversity
    unique_words = set(words)
    ttr = len(unique_words) / n_words
    
    # Hapax Legomenon Ratio — words appearing exactly once
    word_counts = Counter(words)
    hapax = sum(1 for w, c in word_counts.items() if c == 1)
    hapax_ratio = hapax / n_words
    
    # Punctuation features
    n_chars = len(text) if len(text) > 0 else 1
    n_question = text.count('?')
    n_exclam = text.count('!')
    n_comma = text.count(',')
    n_period = text.count('.')
    punct_density = sum(1 for c in text if c in string.punctuation) / n_chars
    
    # Sentence count
    sentences = sent_tokenize(text)
    n_sentences = len(sentences) if len(sentences) > 0 else 1
    avg_sent_len = n_words / n_sentences
    
    return pd.Series({
        'ttr': ttr,
        'hapax_ratio': hapax_ratio,
        'n_question_marks': n_question,
        'n_exclamation_marks': n_exclam,
        'n_commas': n_comma,
        'punct_density': punct_density,
        'avg_sentence_length': avg_sent_len,
    })

def compute_pos_features(text):
    words = str(text).split()[:200]  # Limit for speed
    if len(words) == 0:
        return pd.Series({'noun_ratio': 0, 'verb_ratio': 0, 'adj_ratio': 0, 'adv_ratio': 0})
    tagged = pos_tag(words)
    n = len(tagged)
    nouns = sum(1 for _, tag in tagged if tag.startswith('NN')) / n
    verbs = sum(1 for _, tag in tagged if tag.startswith('VB')) / n
    adjs = sum(1 for _, tag in tagged if tag.startswith('JJ')) / n
    advs = sum(1 for _, tag in tagged if tag.startswith('RB')) / n
    return pd.Series({'noun_ratio': nouns, 'verb_ratio': verbs, 'adj_ratio': adjs, 'adv_ratio': advs})

# Apply to message_to_gpt
print("Extracting lexical & POS features from message_to_gpt...")
lex_to = text_to_gpt.apply(compute_lexical_features)
lex_to.columns = ['msg_to_' + c for c in lex_to.columns]
pos_to = text_to_gpt.apply(compute_pos_features)
pos_to.columns = ['msg_to_' + c for c in pos_to.columns]

# Apply to message_from_gpt
print("Extracting lexical & POS features from message_from_gpt...")
lex_from = text_from_gpt.apply(compute_lexical_features)
lex_from.columns = ['msg_from_' + c for c in lex_from.columns]
pos_from = text_from_gpt.apply(compute_pos_features)
pos_from.columns = ['msg_from_' + c for c in pos_from.columns]

# Merge all
for feat_df in [lex_to, pos_to, lex_from, pos_from]:
    feat_df.index = df.index
    df = pd.concat([df, feat_df], axis=1)

print(f"\n✅ Added {lex_to.shape[1] + pos_to.shape[1] + lex_from.shape[1] + pos_from.shape[1]} lexical/linguistic features")
print(f"DataFrame shape: {df.shape}")
df[['msg_to_ttr', 'msg_to_hapax_ratio', 'msg_to_noun_ratio', 'msg_to_verb_ratio',
    'msg_to_punct_density', 'msg_to_n_question_marks']].describe()

### 4.7 DistilBERT Contextual Embeddings
We use **DistilBERT** (a lighter, faster version of BERT) to extract deep contextual embeddings from the text.
These embeddings capture semantic meaning that TF-IDF and BoW cannot, encoding relationships between words in context.
We keep the full 768-dimensional `[CLS]` embedding for each text column to preserve fine-grained semantic signal.

> ⚠️ This cell uses GPU acceleration. Make sure to enable GPU in Colab: *Runtime → Change runtime type → GPU*

In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel

# Load DistilBERT model and tokenizer
print("Loading DistilBERT model...")
tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_bert = model_bert.to(device)
model_bert.eval()
print(f"✅ DistilBERT loaded on {device}")

def get_bert_embedding(text, max_length=128):
    text = str(text)[:512]  # Truncate very long texts
    encoded = tokenizer_bert(text, return_tensors='pt', padding='max_length',
                             truncation=True, max_length=max_length)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model_bert(**encoded)
    # Use [CLS] token embedding (first token)
    cls_embedding = output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return cls_embedding

# Extract embeddings for message_to_gpt
print("Extracting DistilBERT embeddings for message_to_gpt...")
bert_to_embeddings = []
for i, text in enumerate(text_to_gpt):
    bert_to_embeddings.append(get_bert_embedding(text))
    if (i + 1) % 200 == 0:
        print(f"  Processed {i+1}/{len(text_to_gpt)} messages...")
bert_to_array = np.array(bert_to_embeddings)

bert_to_df = pd.DataFrame(bert_to_array, columns=[f'bert_to_{i}' for i in range(bert_to_array.shape[1])], index=df.index)
print(f"message_to_gpt: kept full {bert_to_array.shape[1]}-dim embedding")

# Extract embeddings for message_from_gpt
print("\nExtracting DistilBERT embeddings for message_from_gpt...")
bert_from_embeddings = []
for i, text in enumerate(text_from_gpt):
    bert_from_embeddings.append(get_bert_embedding(text))
    if (i + 1) % 200 == 0:
        print(f"  Processed {i+1}/{len(text_from_gpt)} messages...")
bert_from_array = np.array(bert_from_embeddings)

bert_from_df = pd.DataFrame(bert_from_array, columns=[f'bert_from_{i}' for i in range(bert_from_array.shape[1])], index=df.index)
print(f"message_from_gpt: kept full {bert_from_array.shape[1]}-dim embedding")

# Add to main dataframe
df = pd.concat([df, bert_to_df, bert_from_df], axis=1)
print(f"\n✅ Added {bert_to_array.shape[1] + bert_from_array.shape[1]} DistilBERT embedding features. DataFrame shape: {df.shape}")

### 4.8 Assemble Final Feature Matrix

In [ ]:
# Separate target from features
y = df['target']
X = df.drop(columns=['target'])

# Fill any remaining NaN values
X = X.fillna(0)

# Ensure all numeric
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)
X = X.astype(np.float64)

print(f"Final feature matrix: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Target distribution: {dict(y.value_counts())}")
print(f"\nFeature categories:")
print(f"  - Numerical (original):    {len([c for c in X.columns if c in ['msg_count_within_p', 'response_time_sec', 'response_time_min', 'words_in_message_to_gpt']])}")
print(f"  - One-hot encoded:         {len([c for c in X.columns if c.startswith('Sex_') or c.startswith('TASK_')])}")
print(f"  - Text length features:    {len([c for c in X.columns if 'char_count' in c or 'word_count' in c or 'avg_word_len' in c or 'word_ratio' in c])}")
print(f"  - Sentiment features:      {len([c for c in X.columns if 'sentiment' in c])}")
print(f"  - Readability features:    {len([c for c in X.columns if 'flesch' in c or 'ari' in c])}")
print(f"  - TF-IDF SVD components:   {len([c for c in X.columns if 'tfidf' in c])}")
print(f"  - NMF topic features:      {len([c for c in X.columns if 'nmf' in c])}")
print(f"  - Lexical/POS features:    {len([c for c in X.columns if 'ttr' in c or 'hapax' in c or 'ratio' in c or 'punct' in c or 'question' in c or 'exclam' in c or 'comma' in c])}")
print(f"  - DistilBERT embeddings:   {len([c for c in X.columns if 'bert' in c])}")

## 5. Feature Selection

### 5.1 Correlation Analysis
We examine Pearson correlations between all features and the target variable.

In [ ]:
# Calculate correlations with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)

# Plot top 25 correlated features
fig, ax = plt.subplots(figsize=(10, 8))
top_corr = correlations.head(25)
colors_corr = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_corr)))
top_corr.plot(kind='barh', ax=ax, color=colors_corr)
ax.set_title('Top 25 Features by Absolute Pearson Correlation with Target', fontsize=14, fontweight='bold')
ax.set_xlabel('|Pearson Correlation|')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop 10 features by correlation:")
for feat, corr in correlations.head(10).items():
    print(f"  {feat:45s} | r = {corr:.4f}")

### 5.2 Mutual Information Scores
Mutual Information captures both linear and non-linear dependencies between features and the target.

In [ ]:
# Calculate Mutual Information scores
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Plot top 25 features by MI
fig, ax = plt.subplots(figsize=(10, 8))
top_mi = mi_series.head(25)
colors_mi = plt.cm.magma(np.linspace(0.2, 0.9, len(top_mi)))
top_mi.plot(kind='barh', ax=ax, color=colors_mi)
ax.set_title('Top 25 Features by Mutual Information Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Mutual Information Score')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Select top features by MI
N_FEATURES = 250
selected_features = mi_series.head(N_FEATURES).index.tolist()
print(f"\n✅ Selected top {N_FEATURES} features by Mutual Information for modeling.")
print(f"Top 10 selected features:")
for i, (feat, score) in enumerate(mi_series.head(10).items()):
    print(f"  {i+1:2d}. {feat:45s} | MI = {score:.4f}")

## 6. Train/Test Split & Scaling

In [ ]:
# Use selected features
X_selected = X[selected_features]

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set:     {X_test_scaled.shape[0]} samples")
print(f"Features:     {X_train_scaled.shape[1]}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

## 7. Model Training & Comparison

### 7.1 Train Multiple Models (including ANN)
We train several classifiers — including a Keras ANN — and compare their performance using 5-fold cross-validation on the training set, then evaluate on the held-out test set.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Define sklearn models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=2000, solver='liblinear'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=200),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'LinearSVC': LinearSVC(random_state=42, dual=False, max_iter=5000),
    'Naive Bayes': GaussianNB(),
}

# Train and evaluate sklearn models
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    results.append({'Model': name, 'CV F1 (mean)': cv_scores.mean(), 'CV F1 (std)': cv_scores.std(),
                    'Test Accuracy': acc, 'Test F1': f1, 'Test Precision': prec, 'Test Recall': rec})
    print(f"  {name:25s} | CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test F1: {f1:.4f} | Test Acc: {acc:.4f}")

# --- ANN Model ---
print("\n  Training ANN (Keras)...")
input_dim = X_train_scaled.shape[1]

def build_ann():
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

ann_model = build_ann()
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
ann_model.fit(X_train_scaled, y_train, epochs=100, batch_size=32,
              validation_split=0.2, callbacks=[early_stop], verbose=0)

y_pred_ann = (ann_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
acc_ann = accuracy_score(y_test, y_pred_ann)
f1_ann = f1_score(y_test, y_pred_ann)
prec_ann = precision_score(y_test, y_pred_ann)
rec_ann = recall_score(y_test, y_pred_ann)
results.append({'Model': 'ANN (Keras)', 'CV F1 (mean)': np.nan, 'CV F1 (std)': np.nan,
                'Test Accuracy': acc_ann, 'Test F1': f1_ann, 'Test Precision': prec_ann, 'Test Recall': rec_ann})
print(f"  {'ANN (Keras)':25s} | Test F1: {f1_ann:.4f} | Test Acc: {acc_ann:.4f}")

results_df = pd.DataFrame(results).sort_values('Test F1', ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("MODEL COMPARISON (sorted by Test F1)")
print("="*80)
results_df

### 7.2 Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 Score comparison
results_sorted = results_df.sort_values('Test F1', ascending=True)
colors_f1 = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(results_sorted)))
axes[0].barh(results_sorted['Model'], results_sorted['Test F1'], color=colors_f1, edgecolor='black', alpha=0.85)
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Test F1 Score by Model', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_sorted['Test F1']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

# Accuracy comparison
axes[1].barh(results_sorted['Model'], results_sorted['Test Accuracy'], color=colors_f1, edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Accuracy')
axes[1].set_title('Test Accuracy by Model', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 1)
for i, v in enumerate(results_sorted['Test Accuracy']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning

### 8.1 Tune Top Models
We tune XGBoost, Gradient Boosting, Logistic Regression, and the ANN with reduced search iterations for faster execution (~5-8 min total).

In [ ]:
tuned_results = []

# --- XGBoost Tuning (reduced: 25 iter × 3cv ≈ 2 min) ---
print("--- Tuning XGBoost ---")
param_dist_xgb = {
    'n_estimators': randint(100, 800),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(3, 10),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 7),
    'reg_alpha': loguniform(0.01, 5),
    'reg_lambda': loguniform(0.01, 5),
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_dist_xgb,
    n_iter=25, scoring='f1', cv=3, random_state=42, n_jobs=-1, verbose=0
)
xgb_search.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {xgb_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_xgb):.4f} | Test Acc: {accuracy_score(y_test, y_pred_xgb):.4f}")
tuned_results.append({'Model': 'XGBoost (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_xgb),
    'Test F1': f1_score(y_test, y_pred_xgb), 'Test Precision': precision_score(y_test, y_pred_xgb),
    'Test Recall': recall_score(y_test, y_pred_xgb), 'estimator': xgb_search.best_estimator_})

# --- Gradient Boosting Tuning (20 iter × 3cv ≈ 2 min) ---
print("\n--- Tuning Gradient Boosting ---")
param_dist_gb = {
    'n_estimators': randint(100, 600),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(3, 8),
    'subsample': uniform(0.6, 0.4),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 8),
}
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=20, scoring='f1', cv=3, random_state=42, n_jobs=-1, verbose=0
)
gb_search.fit(X_train_scaled, y_train)
y_pred_gb = gb_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {gb_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_gb):.4f} | Test Acc: {accuracy_score(y_test, y_pred_gb):.4f}")
tuned_results.append({'Model': 'Gradient Boosting (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_gb),
    'Test F1': f1_score(y_test, y_pred_gb), 'Test Precision': precision_score(y_test, y_pred_gb),
    'Test Recall': recall_score(y_test, y_pred_gb), 'estimator': gb_search.best_estimator_})

# --- Logistic Regression Tuning (15 iter × 3cv ≈ 30 sec) ---
print("\n--- Tuning Logistic Regression ---")
param_dist_lr = [
    {'C': loguniform(0.001, 100), 'solver': ['liblinear'], 'penalty': ['l1', 'l2']},
    {'C': loguniform(0.001, 100), 'solver': ['lbfgs'], 'penalty': ['l2']},
]
lr_search = RandomizedSearchCV(
    LogisticRegression(random_state=42, max_iter=5000),
    param_distributions=param_dist_lr,
    n_iter=15, scoring='f1', cv=3, random_state=42, n_jobs=-1, verbose=0
)
lr_search.fit(X_train_scaled, y_train)
y_pred_lr = lr_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {lr_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_lr):.4f}")
tuned_results.append({'Model': 'Logistic Regression (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_lr),
    'Test F1': f1_score(y_test, y_pred_lr), 'Test Precision': precision_score(y_test, y_pred_lr),
    'Test Recall': recall_score(y_test, y_pred_lr), 'estimator': lr_search.best_estimator_})

# --- ANN Tuning (manual grid: test different architectures) ---
print("\n--- Tuning ANN (Keras) ---")
best_ann_f1 = 0
best_ann_model = None

ann_configs = [
    {'layers': [128, 64, 32], 'dropout': 0.3, 'lr': 0.001},
    {'layers': [256, 128, 64], 'dropout': 0.4, 'lr': 0.0005},
    {'layers': [64, 32], 'dropout': 0.2, 'lr': 0.001},
    {'layers': [128, 64, 32, 16], 'dropout': 0.35, 'lr': 0.0008},
]

for i, cfg in enumerate(ann_configs):
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(input_dim,))])
    for units in cfg['layers']:
        m.add(Dense(units, activation='relu'))
        m.add(BatchNormalization())
        m.add(Dropout(cfg['dropout']))
    m.add(Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
              loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_train_scaled, y_train, epochs=100, batch_size=32,
          validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)], verbose=0)
    y_p = (m.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
    f1_val = f1_score(y_test, y_p)
    print(f"  Config {i+1} {cfg['layers']} dr={cfg['dropout']} lr={cfg['lr']} → F1: {f1_val:.4f}")
    if f1_val > best_ann_f1:
        best_ann_f1 = f1_val
        best_ann_model = m
        best_ann_pred = y_p

tuned_results.append({'Model': 'ANN (tuned)', 'Test Accuracy': accuracy_score(y_test, best_ann_pred),
    'Test F1': f1_score(y_test, best_ann_pred), 'Test Precision': precision_score(y_test, best_ann_pred),
    'Test Recall': recall_score(y_test, best_ann_pred), 'estimator': best_ann_model})
print(f"  Best ANN F1: {best_ann_f1:.4f}")

tuned_df = pd.DataFrame(tuned_results).sort_values('Test F1', ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("TUNED MODEL COMPARISON")
print("="*80)
tuned_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']]

## 9. Final Evaluation

### 9.1 Select Best Model & Classification Report

In [ ]:
# Select the best overall model
best_row = tuned_df.iloc[0]
best_model = best_row['estimator']
best_model_name = best_row['Model']
y_pred_best = best_model.predict(X_test_scaled)

print(f"🏆 Best Model: {best_model_name}")
print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print('='*60)
print(classification_report(y_test, y_pred_best, target_names=['Young Adults', 'Older Adults']))

### 9.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Normalized Confusion Matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[1].set_title(f'Normalized Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

### 9.3 Feature Importance (if tree-based model)

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=selected_features).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = 20
    top_imp = importances.head(top_n).sort_values()
    colors_imp = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_imp)))
    top_imp.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='black')
    ax.set_title(f'Top {top_n} Feature Importances — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 most important features:")
    for i, (feat, imp) in enumerate(importances.head(10).items()):
        print(f"  {i+1:2d}. {feat:45s} | {imp:.4f}")
elif hasattr(best_model, 'coef_'):
    importances = pd.Series(np.abs(best_model.coef_[0]), index=selected_features).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 8))
    top_imp = importances.head(20).sort_values()
    top_imp.plot(kind='barh', ax=ax, color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)), edgecolor='black')
    ax.set_title(f'Top 20 Feature Coefficients (abs) — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('|Coefficient|')
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type.")

## 10. Summary & Conclusions

In [ ]:
# Final comprehensive comparison table
print("="*80)
print("FINAL MODEL COMPARISON TABLE")
print("="*80)

# Combine base + tuned results
all_results = pd.concat([results_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']],
                          tuned_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']]])
all_results = all_results.sort_values('Test F1', ascending=False).reset_index(drop=True)
print(all_results.to_string())

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy:  {best_row['Test Accuracy']:.4f}")
print(f"   F1 Score:  {best_row['Test F1']:.4f}")
print(f"   Precision: {best_row['Test Precision']:.4f}")
print(f"   Recall:    {best_row['Test Recall']:.4f}")
print(f"{'='*80}")

### Key Findings & Discussion

**Dataset Context:**
- The dataset contains 1,275 user-LLM conversation records with demographic labels
- **Target**: `subject_group` – Young Adults vs Older Adults (fairly balanced: ~52% vs 48%)
- The `Age` feature was **excluded** because it's essentially a label proxy (r ≈ 0.97)

**Feature Engineering (NLP-heavy approach):**
- We extracted **~150+ features** from the raw data using established NLP techniques:
  - Basic text statistics (word/character counts, ratios)
  - **VADER Sentiment Analysis** (compound, positive, negative scores)
  - **Readability metrics** (Flesch-Kincaid grade, ARI, Flesch Reading Ease)
  - **Lexical diversity** (Type-Token Ratio, Hapax Legomenon ratio)
  - **POS tag distributions** (noun/verb/adjective/adverb ratios via NLTK POS tagger)
  - **Punctuation & style features** (question marks, exclamation marks, punctuation density)
  - **TF-IDF + SVD** (Latent Semantic Analysis – 40 components)
  - **NMF topic modeling** (20 interpretable topics)
  - **DistilBERT contextual embeddings** (full 768-dim per text column = 1,536 features) — captures deep semantic meaning
  - One-hot encoded categorical features (Sex, TASK)
- Top 250 features were selected using Mutual Information

**Model Performance:**
- Without the `Age` feature, this is a **challenging classification task** since the remaining features have weak individual correlations with the target
- Ensemble tree-based methods (XGBoost, Gradient Boosting, Random Forest) generally outperformed linear models
- Hyperparameter tuning with RandomizedSearchCV provided improvements

**Suggestions for Further Improvement:**
1. **Subject-level aggregation** – aggregate message-level features per subject before training
2. **Ensemble/stacking** – stack or blend the top models
3. **Fine-tuned DistilBERT** – fine-tune the transformer end-to-end on the classification task
4. **Cross-validation with group-based splits** – ensure all messages from one subject are in the same fold
5. **Additional linguistic features** – dependency tree depth, lexical sophistication indices